# Building a Python TutorBot with LangChain and OpenRouter

> **⚠️ First time using colab?**
>
> _Just click "Run All", wait 30s while each block runs and at the end of the file you will see a UI to use the TutorBot_

## Introduction

In this notebook, we build a simple interactive Python tutor chatbot powered by **LangChain** and **OpenRouter**.

The main goal of the project is to create a tutor that can:
- answer Python-related questions
- explain concepts in a short and direct way
- and use a tool to generate practice exercises when the user asks for one

This notebook is organized step by step so that each section explains **what the code does**, **why it is needed**, and **how it contributes to the final chatbot workflow**.


## Step 1 — Importing the Required Libraries

Before building the chatbot, we import the main libraries used throughout the project.

These imports cover four responsibilities:
- connecting to the OpenRouter model
- creating tools with LangChain
- sending tool outputs back to the model
- and managing environment variables securely


In [ ]:
%pip install -q langchain-openrouter langchain python-dotenv gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.1/300.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from google.colab import userdata
from dotenv import load_dotenv
import os
import getpass
import gradio as gr

This block prepares the notebook environment.

Here is the role of each import:
- `ChatOpenRouter` connects the notebook to an LLM served through OpenRouter.
- `tool` allows us to define a callable tool that the model can use.
- `ToolMessage` is used to send the tool's result back into the conversation.
- `os` helps manage environment variables.
- `getpass` lets the user enter their API key securely without printing it on screen.


## Step 2 — Loading the API Key

To use OpenRouter, the notebook needs an API key. Since Google Colab limit the secrets configuration the key is written directly into the code, to help visitors to test this project. This is not a good practice in a production enviroment, but in this POC the API KEY is limit by configuration in OpenRouter to only allow free models.

In [ ]:
# For security reasons this API key was created with a temporary email and it is configured to avoid any credit spending and only supports free models.
os.environ["OPENROUTER_API_KEY"] = "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"

This is a safer approach than hardcoding the API key directly into the notebook.

Why this matters:
- it protects credentials,
- it makes the notebook easier to share,
- and it keeps sensitive information out of the source code.


## Step 3 — Initializing the Language Model

Now we configure the model that will act as the tutor.
In this case, the notebook uses an OpenRouter model through LangChain and sets a few important generation parameters.


In [ ]:
model = ChatOpenRouter(
    model="openrouter/free",
    temperature=0.3,
    max_tokens=1024,
    max_retries=2
)

This block creates the chatbot model instance.

Main parameters:
- `model` selects which model will be used.
- `temperature=0.3` keeps responses more stable and less random.
- `max_tokens=1024` limits how long the model can answer.
- `max_retries=2` gives the request a small safety margin if something fails.

This setup is appropriate for a tutor bot because it favors more controlled and consistent answers.


## Step 4 — Defining a Tool for Practice Exercises

One of the key features of this project is that the tutor can call a tool when the user asks for a Python exercise.

This section defines that tool.


In [ ]:
import requests

@tool
def search_examples(topic: str) -> str:
    """Searches Stack Overflow for Python examples related to the given topic."""
    response = requests.get(
        "https://api.stackexchange.com/2.3/search/excerpts",
        params={
            "order": "desc",
            "sort": "relevance",
            "q": f"Python {topic}",
            "site": "stackoverflow",
            "pagesize": 3,
            "filter": "default"
        }
    )
    data = response.json()

    if not data.get("items"):
        return f"No examples found for '{topic}'."

    results = []
    for item in data["items"]:
        results.append(f"Title: {item['title']}\nExcerpt: {item['excerpt']}")

    return "\n---\n".join(results)

This function is decorated with `@tool`, which tells LangChain that the model is allowed to use it.

What the tool does:
- receives a `topic` and a `difficulty level`,
- and returns a short exercise prompt based on those two inputs.

Even though this is a simple tool, it demonstrates a powerful idea: the model is not limited to plain text generation, because it can also call external logic when needed.


## Step 5 — Registering the Tool and Binding it to the Model

After defining the tool, we need to make it available to the model.
This step stores the tool in a list, creates a dictionary for easier access, and binds the tool to the chatbot.


In [ ]:
tools = [search_examples]
tools_dict = {tool.name: tool for tool in tools}

model_with_tools = model.bind_tools(tools)

This block connects the tool system to the language model.

Why each part matters:
- `tools` stores all tools in a list.
- `tools_dict` makes it easy to find and execute a tool by name.
- `model.bind_tools(tools)` enables tool-calling behavior in the model.

Without this step, the chatbot would know nothing about the exercise generator.


## Step 6 — Writing the System Prompt

The system prompt defines the chatbot's role, tone, and behavior.
You asked for the prompt to be written in English, so the instruction below is fully written in English while still telling the assistant to answer the user in Spanish.


In [ ]:
system_prompt = """
ROLE
You are TutorBot, a programming tutor specialized in Python.
Your students are beginners learning Python for the first time.

INSTRUCTIONS
- Explain concepts in simple terms. Use analogies when they help.
- When a student makes a mistake, don't just give the answer. Point out where the error is and guide them to fix it themselves.
- Keep responses short. If a topic is complex, break it into steps and go one at a time.
- When explaining syntax, always include a short code example.
- If the student asks something outside of Python, politely redirect them.
- Add a source link if you used the search_example tool.

CRITICAL CONSTRAINTS
- DO NOT write a complete solution to an exercise you gave. Give hints, not answers. UNLESS the student is stuck after 2-3 hints.
- DO NOT invent library names or functions that don't exist.
- DO NOT answer questions outside Computer Science topics.

LANGUAGE
Always use the same language than the USER messages.

TOOLS
- search_examples: Searches Stack Overflow for real Python questions and code snippets related to a topic. Use it when:
  - The student asks "how do people usually do X?"
  - You want to show a real-world usage pattern, not just a textbook example.
  - The student is confused and seeing how others solved a similar problem would help.
  - The question from the user require a more deep understanding or it's a edge case.

EXAMPLES
User: "What is a list?"
You: Explain it directly. No need to search.

User: "How do people usually read CSV files in Python?"
You: Use search_examples to find real patterns, then summarize the most common approach.
"""

This prompt controls the chatbot's behavior.

It establishes four core rules:
- the assistant is a Python tutor
- responses should be in Spanish
- explanations should be short and direct
- and the model is allowed to use the `generate_exercise` tool when appropriate.

This is the main instruction layer that shapes the tutor's personality and response style.


## Step 7 — Starting the Conversation Memory

Before the interaction begins, we initialize the message list with the system prompt.
This list stores the conversation history between the user, the model, and any tool calls.

This is important because the model does not remember previous turns on its own.
Instead, it relies on the `messages` list to understand the context of the conversation and respond consistently.


In [ ]:
# Start the conversation history with the system prompt.
# This gives the model its initial behavior and provides the foundation
# for storing future user messages, assistant replies, and tool results.
messages = [
    ("system", system_prompt)
]

This message list is essential because the model needs conversation context to respond correctly.

At the beginning, the only message stored is the system message.
As the user interacts with the bot, new messages will be appended to this list.

In other words, this step creates the chatbot's short-term memory inside the notebook.


## Step 8 — Limiting API Calls per Session

To make testing safer and more controlled, we define a maximum number of model calls allowed in a single Colab session.
This prevents accidental overuse of the API while the notebook is being tested.


In [ ]:
MAX_CALLS_PER_SESSION = 15

if "session_call_count" not in globals():
    session_call_count = 0

## Step 9 — Running the Interactive TutorBot

This final section launches the chatbot loop and applies the full runtime logic of the project.

Here is the workflow:

1. The notebook waits for user input.
2. If the user types `exit`, `quit`, or `cancel`, the program stops.
3. Otherwise, the input is added to the conversation history.
4. Before calling the model, the notebook checks whether the session API-call limit has been reached.
5. If the limit has not been reached, the model is invoked and the session counter is increased.
6. If the model requests a tool, the notebook executes the tool and stores the result as a `ToolMessage`.
7. Before generating the final answer after the tool call, the notebook checks the session limit again.
8. The model then produces the final response for the user.
9. The notebook keeps only the most recent messages to avoid unnecessary context growth.

This section combines user interaction, session-level API-call control, tool execution, and short-term memory management in a single runtime loop.


In [ ]:
def tutorbot_response(message, history):
    global session_call_count

    try:
        if message.lower().strip() in ["exit", "quit", "cancel"]:
            return "Goodbye."

        messages = [SystemMessage(content=system_prompt)]

        for user_msg, bot_msg in history:
            if user_msg:
                messages.append(HumanMessage(content=user_msg))
            if bot_msg:
                messages.append(AIMessage(content=bot_msg))

        messages.append(HumanMessage(content=message))

        if session_call_count >= MAX_CALLS_PER_SESSION:
            return "You have reached the maximum number of API calls allowed in this session."

        session_call_count += 1
        ai_msg = model_with_tools.invoke(messages)

        if ai_msg.tool_calls:
            messages.append(ai_msg)

            for tool_call in ai_msg.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]

                tool_result = tools_dict[tool_name].invoke(tool_args)

                messages.append(
                    ToolMessage(
                        content=tool_result,
                        tool_call_id=tool_call["id"]
                    )
                )

            if session_call_count >= MAX_CALLS_PER_SESSION:
                return "You have reached the maximum number of API calls allowed in this session."

            session_call_count += 1
            final_msg = model_with_tools.invoke(messages)
            return final_msg.content

        return ai_msg.content

    except Exception as e:
        return f"An error occurred: {e}"

## Step 10 - Gradio


In [ ]:
demo = gr.ChatInterface(
    fn=tutorbot_response,
    title="Python TutorBot",
    description="Ask Python questions, request explanations, or generate exercises.",
    examples=[
        "Explain what a while loop is.",
        "Give me an easy exercise about lists.",
        "What is the difference between a list and a tuple?"
    ],
)

demo.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0d313da06a9707a185.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Conclusion

In this notebook, we built a small but functional Python TutorBot using **LangChain**, **OpenRouter**, and a custom **tool**.

The project includes:
- secure API key handling
- model initialization
- a custom exercise-generation tool
- tool binding
- a system prompt in English
- session-level API-call limiting for safer testing
- and an interactive chat loop with short-term conversation memory.

This structure makes the notebook easier to understand, present, and reuse. It also gives the project a more professional format for learning, demonstration, and portfolio purposes.
